# Алгоритмы: описания, реализации и тестирование на Python

В этой тетрадке рассмотрены следующие алгоритмы:

1. **Бинарный поиск** (в отсортированном массиве);
2. **Вещественный бинарный поиск** (поиск корня функции с заданной точностью);
3. **Сортировка подсчётом через список** (пример данных — степени 10);
4. **Сортировка подсчётом через словарь** (пример данных — степени 10);
5. **Нерекурсивная сортировка слиянием (Merge Sort)** с вынесенной функцией `merge`.

Каждая реализация снабжена комментариями и тестируется с помощью `assert`.
Если ячейка выполнилась без ошибок — все тесты пройдены.

## 1. Бинарный поиск (Binary Search)

**Идея:** ищем элемент `target` в **отсортированном** массиве, на каждом шаге отбрасывая половину оставшегося интервала:

1. Смотрим на средний элемент интервала `[left, right]`.
2. Если он равен искомому — готово.
3. Если он меньше искомого — продолжаем поиск в правой половине.
4. Иначе — в левой.

**Сложность:** `O(log n)` по времени, `O(1)` по памяти.  
**Предусловие:** массив должен быть отсортирован.

In [ ]:
def binary_search(arr, target):
    '''
    Бинарный поиск элемента в отсортированном массиве.

    Параметры:
        arr    -- отсортированный по возрастанию список;
        target -- искомый элемент.
    Возвращает:
        индекс найденного элемента или -1, если элемент отсутствует.
    '''
    left, right = 0, len(arr) - 1      # границы интервала поиска

    while left <= right:               # пока интервал поиска непустой
        mid = (left + right) // 2      # индекс середины интервала

        if arr[mid] == target:         # элемент найден — возвращаем индекс
            return mid
        elif arr[mid] < target:        # искомый элемент правее середины
            left = mid + 1
        else:                          # искомый элемент левее середины
            right = mid - 1

    return -1                          # интервал пуст — элемента нет в массиве


# --- Тестирование ---
data = [1, 3, 5, 7, 9, 11, 13, 15]

# присутствующие элементы
assert binary_search(data, 1) == 0               # первый элемент
assert binary_search(data, 7) == 3               # элемент в середине
assert binary_search(data, 15) == len(data) - 1  # последний элемент

# отсутствующие элементы
assert binary_search(data, 0) == -1
assert binary_search(data, 8) == -1
assert binary_search(data, 16) == -1

# граничные случаи
assert binary_search([], 5) == -1       # пустой массив
assert binary_search([42], 42) == 0     # один элемент, найден
assert binary_search([42], 7) == -1     # один элемент, не найден

print('Бинарный поиск: все тесты пройдены')

## 2. Вещественный бинарный поиск (поиск корня функции)

**Идея:** пусть непрерывная функция `f` меняет знак на отрезке `[lo, hi]`,
то есть `f(lo) * f(hi) < 0`. Тогда на этом отрезке существует корень.
На каждой итерации делим отрезок пополам и оставляем ту половину,
на которой знак продолжает меняться.

**Сложность:** `O(log((hi - lo) / eps))` итераций для достижения точности `eps`.  
**Применение:** вычисление квадратных и кубических корней, решение монотонных уравнений.

In [ ]:
import math


def real_binary_search(f, lo, hi, eps=1e-10, max_iter=1000):
    '''
    Вещественный бинарный поиск корня непрерывной функции.

    Ищет такой x, что f(x) = 0, на отрезке [lo, hi],
    на концах которого функция меняет знак: f(lo) * f(hi) < 0.

    Параметры:
        f        -- непрерывная функция;
        lo, hi   -- границы отрезка со сменой знака;
        eps      -- требуемая точность;
        max_iter -- максимальное число итераций.
    '''
    f_lo = f(lo)                       # значение функции на левом конце отрезка

    for _ in range(max_iter):
        if hi - lo <= 2 * eps:         # достигли требуемой точности — останавливаемся
            break

        mid = (lo + hi) / 2            # середина отрезка
        f_mid = f(mid)

        if f_lo * f_mid <= 0:          # знак меняется на левой половине
            hi = mid
        else:                          # иначе корень лежит в правой половине
            lo = mid
            f_lo = f_mid

    return (lo + hi) / 2               # приближение корня (погрешность <= eps)


# --- Тестирование ---

# квадратный корень из 2: решаем уравнение x^2 - 2 = 0 на отрезке [0, 2]
root2 = real_binary_search(lambda x: x * x - 2, 0.0, 2.0)
assert abs(root2 - math.sqrt(2)) < 1e-6, root2

# квадратный корень из 9 (целый ответ)
root9 = real_binary_search(lambda x: x * x - 9, 0.0, 10.0)
assert abs(root9 - 3.0) < 1e-6, root9

# кубический корень из 27: решаем x^3 - 27 = 0
cbrt27 = real_binary_search(lambda x: x ** 3 - 27, 0.0, 100.0)
assert abs(cbrt27 - 3.0) < 1e-6, cbrt27

# отрицательный корень: уравнение x^2 - 4 = 0 на отрезке [-5, 0] -> ответ -2
neg_root = real_binary_search(lambda x: x * x - 4, -5.0, 0.0)
assert abs(neg_root - (-2.0)) < 1e-6, neg_root

# убывающая функция: 0.5 - 2^(-x) = 0  =>  x = 1
dec_root = real_binary_search(lambda x: 0.5 - 2 ** (-x), 0.0, 10.0)
assert abs(dec_root - 1.0) < 1e-6, dec_root

print('Вещественный бинарный поиск: все тесты пройдены')

## 3. Сортировка подсчётом (Counting Sort)

**Идея:** подсчитываем, сколько раз встречается каждое значение,
а затем выводим значения по возрастанию с учётом их количества.

**Пример данных:** массив из степеней 10: `1, 10, 100, 1000`, записанных вперемешку.
Различных значений всего 4, но диапазон велик (от 1 до 1000) —
поэтому интересно сравнить две реализации:

- **через список** — классический вариант: список счётчиков длины `max - min + 1`.
  Работает за `O(n + k)`, но тратит память на «пустые» значения диапазона;
- **через словарь** — счётчики хранятся только для реально встречающихся значений,
  что экономит память на разреженных данных.

In [ ]:
# Пример данных: степени 10 (1, 10, 100, 1000), записанные вперемешку.
# Эта переменная будет использоваться и в следующих ячейках.
POWERS_DATA = [100, 10, 1, 1000, 10, 1, 100, 10, 1, 1000, 100]


def counting_sort_list(arr):
    '''
    Сортировка подсчётом, реализованная через СПИСОК счётчиков.

    Счётчики хранятся в списке длины (max - min + 1),
    где индекс соответствует значению (со сдвигом на min).

    Сложность: O(n + k) по времени и памяти,
    где k = max(arr) - min(arr) + 1 -- размер диапазона значений.
    '''
    if not arr:
        return []

    mn, mx = min(arr), max(arr)            # границы диапазона значений
    counts = [0] * (mx - mn + 1)           # список счётчиков для каждого значения диапазона

    for x in arr:                          # подсчитываем частоты значений
        counts[x - mn] += 1

    result = []
    for offset, cnt in enumerate(counts):  # восстанавливаем значения по возрастанию
        if cnt:
            result.extend([offset + mn] * cnt)

    return result


# --- Тестирование ---
expected = [1, 1, 1, 10, 10, 10, 100, 100, 100, 1000, 1000]

assert counting_sort_list(POWERS_DATA) == expected
assert counting_sort_list(POWERS_DATA) == sorted(POWERS_DATA)
assert counting_sort_list([]) == []                     # пустой массив
assert counting_sort_list([10]) == [10]                 # один элемент
assert counting_sort_list([100, 100, 100]) == [100, 100, 100]

# обратите внимание: список счётчиков имеет длину 1000,
# хотя различных значений в массиве всего 4 (1, 10, 100, 1000)
mn, mx = min(POWERS_DATA), max(POWERS_DATA)
assert mx - mn + 1 == 1000
assert len(set(POWERS_DATA)) == 4

print('Сортировка подсчётом (список): все тесты пройдены')

In [ ]:
def counting_sort_dict(arr):
    '''
    Сортировка подсчётом, реализованная через СЛОВАРЬ счётчиков.

    В словаре хранятся только реально встречающиеся значения,
    поэтому на разреженных данных (как степени 10) памяти тратится
    существенно меньше, чем в варианте со списком.

    Сложность: O(n + m log m), где m -- число различных значений
    (слагаемое m log m даёт сортировка ключей словаря).
    '''
    counts = {}                        # словарь: значение -> частота его появления

    for x in arr:                      # подсчитываем частоты значений
        counts[x] = counts.get(x, 0) + 1

    result = []
    for key in sorted(counts):         # идём по ключам в порядке возрастания
        result.extend([key] * counts[key])

    return result


# --- Тестирование ---
expected = [1, 1, 1, 10, 10, 10, 100, 100, 100, 1000, 1000]

assert counting_sort_dict(POWERS_DATA) == expected
assert counting_sort_dict(POWERS_DATA) == sorted(POWERS_DATA)
assert counting_sort_dict([]) == []
assert counting_sort_dict([10]) == [10]
assert counting_sort_dict([100, 100, 100]) == [100, 100, 100]

# словарю счётчиков нужно всего 4 записи вместо 1000 у варианта со списком
tmp_counts = {}
for x in POWERS_DATA:
    tmp_counts[x] = tmp_counts.get(x, 0) + 1
assert len(tmp_counts) == 4

print('Сортировка подсчётом (словарь): все тесты пройдены')

## 4. Нерекурсивная сортировка слиянием (итеративный Merge Sort)

**Идея (снизу вверх, без рекурсии):**

1. Считаем массив последовательностью отсортированных блоков длины 1.
2. На каждом проходе сливаем соседние блоки: получаем блоки длины 2, затем 4, 8 и т. д.,
   пока весь массив не станет одним отсортированным блоком.
3. Функция `merge` вынесена отдельно: она сливает два отсортированных участка
   `src[start:mid]` и `src[mid:end]` в буфер `dst`.

**Сложность:** `O(n log n)` по времени, `O(n)` дополнительной памяти.  
Сортировка **устойчива** (равные элементы сохраняют исходный порядок),
так как в `merge` при равенстве берётся элемент из левой половины (условие `<=`).

In [ ]:
def merge(src, dst, start, mid, end):
    '''
    Слияние двух отсортированных участков массива.

    Сливает отсортированные участки src[start:mid] и src[mid:end]
    в массив-буфер dst на позиции [start, end).
    '''
    i, j, k = start, mid, start   # указатели: левая половина, правая половина, позиция записи

    # сравниваем «головы» обеих половин и записываем меньший элемент
    while i < mid and j < end:
        if src[i] <= src[j]:      # <= обеспечивает устойчивость сортировки
            dst[k] = src[i]
            i += 1
        else:
            dst[k] = src[j]
            j += 1
        k += 1

    # дописываем хвосты (выполнится не более одного из циклов ниже)
    while i < mid:
        dst[k] = src[i]
        i += 1
        k += 1

    while j < end:
        dst[k] = src[j]
        j += 1
        k += 1


def merge_sort(arr):
    '''
    Итеративная (нерекурсивная) сортировка слиянием «снизу вверх».

    На каждом проходе соседние блоки длины width сливаются попарно в блоки длины 2 * width.
    Размер блока удваивается: 1, 2, 4, 8, ... пока не покроет весь массив.
    Рекурсия не используется, поэтому стек вызовов не переполняется.

    Сложность: O(n log n) по времени, O(n) дополнительной памяти.
    '''
    n = len(arr)
    if n <= 1:
        return arr[:]

    src = arr[:]           # массив-источник (входной список не изменяем)
    dst = [0] * n          # буфер для результатов слияния

    width = 1              # текущий размер сливаемых блоков
    while width < n:
        for start in range(0, n, 2 * width):
            mid = min(start + width, n)        # конец левой половины (он же начало правой)
            end = min(start + 2 * width, n)    # конец правой половины (обрезается краем массива)
            merge(src, dst, start, mid, end)   # сливаем пару блоков в буфер
        src, dst = dst, src    # буфер становится источником для следующего прохода
        width *= 2

    return src


# --- Тестирование ---
import random

# базовые проверки
assert merge_sort([5, 2, 4, 6, 1, 3]) == [1, 2, 3, 4, 5, 6]
assert merge_sort([]) == []                              # пустой массив
assert merge_sort([7]) == [7]                            # один элемент
assert merge_sort([3, 3, 3, 3]) == [3, 3, 3, 3]          # все элементы равны
assert merge_sort([9, 8, 7, 6, 5, 4]) == [4, 5, 6, 7, 8, 9]   # обратный порядок
assert merge_sort([-5, 10, -3, 0, 7]) == [-5, -3, 0, 7, 10]   # отрицательные числа

# те же степени 10, что и в сортировке подсчётом (нечётная длина массива)
assert merge_sort(POWERS_DATA) == sorted(POWERS_DATA)

# случайные тесты против встроенной сортировки
random.seed(42)
for _ in range(30):
    arr = [random.randint(-100, 100) for _ in range(random.randint(0, 50))]
    assert merge_sort(arr) == sorted(arr), f'Ошибка на массиве: {arr}'

# исходный массив не должен изменяться функцией сортировки
original = [3, 1, 2]
assert merge_sort(original) == [1, 2, 3]
assert original == [3, 1, 2]

print('Итеративная сортировка слиянием: все тесты пройдены')

## Итоги

| Алгоритм | Время | Память | Примечание |
|---|---|---|---|
| Бинарный поиск | `O(log n)` | `O(1)` | требует отсортированного массива |
| Вещественный бинарный поиск | `O(log((hi - lo) / eps))` | `O(1)` | функция должна менять знак на отрезке |
| Сортировка подсчётом (список) | `O(n + k)` | `O(n + k)` | `k` — размер диапазона значений |
| Сортировка подсчётом (словарь) | `O(n + m log m)` | `O(n + m)` | `m` — число различных значений |
| Итеративный Merge Sort | `O(n log n)` | `O(n)` | устойчивая, без рекурсии |

Все реализации протестированы с помощью `assert`: если ячейки выполнились без ошибок — тесты прошли успешно.